# Plot configuration iterated during the Bayesian Optimization process

In [9]:
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# 1️⃣ Load JSON from the result json file
result_path = "/home/chenzhil/maplecg_nfs/BO-for-LLMs/results/arc_challenge/llama-8b_EI_arc_challenge__mixed_acq_EI_eval_eval_loss_bo_data.json"
with open(result_path, "r") as f:
    raw = json.load(f)

# 2️⃣ Extract metadata
info = raw["final_info_stored"]
domain_names = info["training domain"]
num_domains = len(domain_names)
run_bo_on = info["command line args"]["run_BO_on"]  # "data", "model", or "both"
inputs = info["inputs_iterated"][0]  # first trial
best_perf = info["best_seen_performance"][0]
T = len(inputs)

MODULE_NAMES = ["q_proj", "v_proj", "up_proj", "down_proj", "gate_proj"]
save_dir = "/home/chenzhil/maplecg_nfs/BO-for-LLMs/plots_and_animation"

# 3️⃣ Parse per-iteration data depending on what was optimized
has_data = run_bo_on in ("data", "both")
has_model = run_bo_on in ("model", "both")

if has_data:
    mix_data = np.array([inp[:num_domains] for inp in inputs])  # (T, num_domains)

if has_model:
    offset = num_domains if run_bo_on == "both" else 0
    num_layers_arr  = [int(inp[offset]) for inp in inputs]
    modules_arr     = np.array([[int(inp[offset + 1 + m]) for m in range(5)] for inp in inputs])
    rank_arr        = [int(inp[-4]) for inp in inputs]
    dropout_arr     = [round(inp[-3], 4) for inp in inputs]
    alpha_arr       = [round(inp[-2], 2) for inp in inputs]
    reverse_arr     = [int(inp[-1]) for inp in inputs]

# 4️⃣ Determine layout: one row per panel
n_rows = has_data + (4 if has_model else 0) + 1  # +1 for performance
fig, axes = plt.subplots(n_rows, 1, figsize=(max(10, num_domains * 1.0), 3.2 * n_rows))
if n_rows == 1:
    axes = [axes]
fig.suptitle(f"BO Configuration Evolution — Iteration 0  [{run_bo_on}]", fontsize=14, y=0.99)
plt.subplots_adjust(hspace=0.6, top=0.94)

ax_idx = 0

# --- Panel: Data mixing ratios ---
bars_mix = None
if has_data:
    ax_mix = axes[ax_idx]; ax_idx += 1
    bars_mix = ax_mix.bar(domain_names, mix_data[0], color="steelblue")
    ax_mix.set_ylim(0, 1)
    ax_mix.set_ylabel("Mixing Ratio")
    ax_mix.set_title("Data Mixture")
    ax_mix.tick_params(axis="x", rotation=45)

# --- Panel: Module mask ---
bars_mod = None
bars_scalar = None
bars_drop = None
bars_rev = None
if has_model:
    # module on/off
    ax_mod = axes[ax_idx]; ax_idx += 1
    bars_mod = ax_mod.bar(MODULE_NAMES, modules_arr[0], color="coral")
    ax_mod.set_ylim(-0.1, 1.5)
    ax_mod.set_yticks([0, 1])
    ax_mod.set_ylabel("On / Off")
    ax_mod.set_title("LoRA Module Mask")

    # scalar params: num_layers, rank, alpha
    scalar_keys = ["num_layers", "rank", "alpha"]
    scalar_arrays = {"num_layers": num_layers_arr, "rank": rank_arr, "alpha": alpha_arr}
    scalar_colors = ["#2ca02c", "#9467bd", "#e377c2"]
    ax_scalar = axes[ax_idx]; ax_idx += 1
    bars_scalar = ax_scalar.bar(scalar_keys, [scalar_arrays[k][0] for k in scalar_keys], color=scalar_colors)
    max_scalar = max(max(scalar_arrays[k]) for k in scalar_keys)
    ax_scalar.set_ylim(0, max_scalar * 1.2 + 1)
    ax_scalar.set_ylabel("Value")
    ax_scalar.set_title("LoRA Params: num_layers / rank / alpha")

    # dropout
    ax_drop = axes[ax_idx]; ax_idx += 1
    bars_drop = ax_drop.bar(["dropout"], [dropout_arr[0]], color="#ff7f0e")
    ax_drop.set_ylim(0, 0.12)
    ax_drop.set_ylabel("Value")
    ax_drop.set_title("LoRA Dropout")

    # reverse
    ax_rev = axes[ax_idx]; ax_idx += 1
    bars_rev = ax_rev.bar(["reverse"], [reverse_arr[0]], color="#8c564b")
    ax_rev.set_ylim(-0.1, 1.5)
    ax_rev.set_yticks([0, 1])
    ax_rev.set_ylabel("On / Off")
    ax_rev.set_title("LoRA Reverse (0=rear layers, 1=front layers)")

# --- Panel: Best performance so far ---
ax_perf = axes[ax_idx]
perf_line, = ax_perf.plot([], [], "o-", color="green", linewidth=2, markersize=4)
ax_perf.set_xlim(0, T - 1)
perf_min, perf_max = min(best_perf[:T]), max(best_perf[:T])
margin = (perf_max - perf_min) * 0.15 + 1e-6
ax_perf.set_ylim(perf_min - margin, perf_max + margin)
ax_perf.set_xlabel("Iteration")
ax_perf.set_ylabel("Performance")
ax_perf.set_title("Best Performance So Far")
ax_perf.grid(True, alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.96])

# 5️⃣ Update function
def update(frame):
    fig.suptitle(f"BO Configuration Evolution — Iteration {frame}  [{run_bo_on}]", fontsize=14, y=0.99)

    if bars_mix is not None:
        for i, bar in enumerate(bars_mix):
            bar.set_height(mix_data[frame, i])

    if bars_mod is not None:
        for i, bar in enumerate(bars_mod):
            bar.set_height(modules_arr[frame, i])

    if bars_scalar is not None:
        for i, k in enumerate(scalar_keys):
            bars_scalar[i].set_height(scalar_arrays[k][frame])

    if bars_drop is not None:
        bars_drop[0].set_height(dropout_arr[frame])

    if bars_rev is not None:
        bars_rev[0].set_height(reverse_arr[frame])

    perf_line.set_data(range(frame + 1), best_perf[:frame + 1])
    return []

# 6️⃣ Create animation and save as GIF
anim = FuncAnimation(fig, update, frames=T, blit=False)
gif_path = f"{save_dir}/bo_config_evolution.gif"
anim.save(gif_path, writer=PillowWriter(fps=2))
plt.close()
print(f"✅ Saved GIF ({T} frames) to {gif_path}")

✅ Saved GIF (2 frames) to /home/chenzhil/maplecg_nfs/BO-for-LLMs/plots_and_animation/bo_config_evolution.gif
